In [1]:
# ============================================================
# STEP 9.1 — TRAINING CONFIGURATION
# ============================================================

import os
import json
import math
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("=" * 70)
print("STEP 9.1 — TRAINING CONFIGURATION")
print("=" * 70)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

BASE_DIR = "data/tokenized"

TRAIN_FILE = os.path.join(BASE_DIR, "train_tokenized.json")
VAL_FILE = os.path.join(BASE_DIR, "validation_tokenized.json")
TEST_FILE = os.path.join(BASE_DIR, "test_tokenized.json")

OUTPUT_DIR = "outputs/model1_qlora"

MAX_LENGTH = 2048

print("Model:", MODEL_NAME)
print("Train:", TRAIN_FILE)
print("Validation:", VAL_FILE)
print("Test:", TEST_FILE)
print("Output:", OUTPUT_DIR)

print()
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

print(
    "BF16 supported:",
    torch.cuda.is_bf16_supported()
)

print("=" * 70)

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0912 10:11:59.563000 17680 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


STEP 9.1 — TRAINING CONFIGURATION
Model: meta-llama/Llama-3.1-8B-Instruct
Train: data/tokenized\train_tokenized.json
Validation: data/tokenized\validation_tokenized.json
Test: data/tokenized\test_tokenized.json
Output: outputs/model1_qlora

PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
GPU memory: 11.94 GB
BF16 supported: True


In [2]:
# ============================================================
# STEP 9.2 — LOAD TOKENIZED DATA
# ============================================================

def load_tokenized_json(path):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    return records


train_records = load_tokenized_json(TRAIN_FILE)
val_records = load_tokenized_json(VAL_FILE)
test_records = load_tokenized_json(TEST_FILE)


print("=" * 70)
print("STEP 9.2 — TOKENIZED DATA LOADED")
print("=" * 70)

print("Train records:", len(train_records))
print("Validation records:", len(val_records))
print("Test records:", len(test_records))


# ------------------------------------------------------------
# DATASET-SIZE-INDEPENDENT SPLIT VERIFICATION
# ------------------------------------------------------------

total_records = (
    len(train_records)
    + len(val_records)
    + len(test_records)
)

expected_train = int(total_records * 0.80)
expected_val = int(total_records * 0.10)
expected_test = total_records - expected_train - expected_val


if len(train_records) != expected_train:
    raise ValueError(
        f"Train size mismatch: "
        f"expected {expected_train}, "
        f"got {len(train_records)}"
    )

if len(val_records) != expected_val:
    raise ValueError(
        f"Validation size mismatch: "
        f"expected {expected_val}, "
        f"got {len(val_records)}"
    )

if len(test_records) != expected_test:
    raise ValueError(
        f"Test size mismatch: "
        f"expected {expected_test}, "
        f"got {len(test_records)}"
    )

print("Dataset-size verification: PASSED")


# ------------------------------------------------------------
# VERIFY ESSENTIAL FIELDS
# ------------------------------------------------------------

required_fields = {
    "id",
    "input_ids",
    "attention_mask",
    "labels",
}

for split_name, records in [
    ("train", train_records),
    ("validation", val_records),
    ("test", test_records),
]:

    for record in records:

        missing = required_fields - set(record.keys())

        if missing:
            raise ValueError(
                f"{split_name} record {record.get('id')} "
                f"is missing fields: {missing}"
            )

        if not (
            len(record["input_ids"])
            == len(record["attention_mask"])
            == len(record["labels"])
        ):
            raise ValueError(
                f"{split_name} record {record.get('id')} "
                f"has inconsistent sequence/label lengths"
            )


print("Required fields: PASSED")
print("Sequence/label lengths: PASSED")

print("=" * 70)

STEP 9.2 — TOKENIZED DATA LOADED
Train records: 600
Validation records: 75
Test records: 75
Dataset-size verification: PASSED
Required fields: PASSED
Sequence/label lengths: PASSED


In [3]:
# ============================================================
# STEP 9.3 — CREATE HUGGING FACE DATASETS
# ============================================================

MODEL_FIELDS = [
    "input_ids",
    "attention_mask",
    "labels",
]

train_dataset = (
    Dataset.from_list(train_records)
    .select_columns(MODEL_FIELDS)
)

val_dataset = (
    Dataset.from_list(val_records)
    .select_columns(MODEL_FIELDS)
)

test_dataset = (
    Dataset.from_list(test_records)
    .select_columns(MODEL_FIELDS)
)


print("=" * 70)
print("STEP 9.3 — DATASETS CREATED")
print("=" * 70)

print("Train:", train_dataset)
print("Validation:", val_dataset)
print("Test:", test_dataset)

print()
print("Train columns:", train_dataset.column_names)
print("Validation columns:", val_dataset.column_names)
print("Test columns:", test_dataset.column_names)


# ------------------------------------------------------------
# DATASET-SIZE-INDEPENDENT VERIFICATION
# ------------------------------------------------------------

total_records = (
    len(train_dataset)
    + len(val_dataset)
    + len(test_dataset)
)

expected_train = int(total_records * 0.80)
expected_val = int(total_records * 0.10)
expected_test = total_records - expected_train - expected_val


assert len(train_dataset) == expected_train, (
    f"Train size mismatch: expected {expected_train}, "
    f"got {len(train_dataset)}"
)

assert len(val_dataset) == expected_val, (
    f"Validation size mismatch: expected {expected_val}, "
    f"got {len(val_dataset)}"
)

assert len(test_dataset) == expected_test, (
    f"Test size mismatch: expected {expected_test}, "
    f"got {len(test_dataset)}"
)


print()
print("Dataset creation: PASSED")
print("=" * 70)

STEP 9.3 — DATASETS CREATED
Train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 600
})
Validation: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 75
})
Test: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 75
})

Train columns: ['input_ids', 'attention_mask', 'labels']
Validation columns: ['input_ids', 'attention_mask', 'labels']
Test columns: ['input_ids', 'attention_mask', 'labels']

Dataset creation: PASSED


In [4]:
# ============================================================
# STEP 9.4 — LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

# Llama 3.1 does not have a separate PAD token by default.
# For batching, use EOS as PAD.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("=" * 70)
print("STEP 9.4 — TOKENIZER")
print("=" * 70)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("PAD ID:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)

assert tokenizer.vocab_size == 128000

print("Tokenizer: PASSED")
print("=" * 70)

STEP 9.4 — TOKENIZER
Tokenizer: TokenizersBackend
Vocabulary size: 128000
PAD token: <|eot_id|>
PAD ID: 128009
EOS token: <|eot_id|>
EOS ID: 128009
Tokenizer: PASSED


In [5]:
# ============================================================
# STEP 9.5 — 4-BIT QLoRA CONFIGURATION
# ============================================================

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("=" * 70)
print("STEP 9.5 — QLoRA CONFIGURATION")
print("=" * 70)

print("4-bit loading: True")
print("Quantization type: NF4")
print("Double quantization: True")
print("Compute dtype:", compute_dtype)

print("=" * 70)

STEP 9.5 — QLoRA CONFIGURATION
4-bit loading: True
Quantization type: NF4
Double quantization: True
Compute dtype: torch.bfloat16


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

Loading weights: 100%|██████████| 291/291 [00:21<00:00, 13.57it/s]


Model loaded successfully!
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [8]:
# STEP 9.7 — Prepare model for QLoRA

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

print("Model prepared for QLoRA.")

Model prepared for QLoRA.


In [9]:
# STEP 9.8 — Add LoRA

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [10]:
from transformers import DataCollatorForSeq2Seq
# STEP 9.9 — Data collator

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

print("Data collator ready.")

Data collator ready.


In [13]:
# STEP 9.10 — Training settings

training_args = TrainingArguments(
    output_dir="outputs/model1_qlora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    gradient_checkpointing=False,

    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_steps=5,

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False
)

print("Training settings ready.")

Training settings ready.


In [14]:
# STEP 9.11 — Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer ready.")

Trainer ready.


In [15]:
# STEP 9.12 — Final check

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory used:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

model.print_trainable_parameters()

print("READY TO TRAIN.")

Train examples: 600
Validation examples: 75
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
GPU memory used: 7.43 GB
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
READY TO TRAIN.


In [16]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.401518,1.398128


TrainOutput(global_step=75, training_loss=1.608747984568278, metrics={'train_runtime': 771.1068, 'train_samples_per_second': 0.778, 'train_steps_per_second': 0.097, 'total_flos': 4793060965097472.0, 'train_loss': 1.608747984568278, 'epoch': 1.0})

In [17]:
# STEP 9.14 — Save Model 1

trainer.save_model("outputs/model1_qlora/final")
tokenizer.save_pretrained("outputs/model1_qlora/final")

print("Model 1 saved successfully.")

Model 1 saved successfully.
